# GridBattle PCG Agent Evaluation Analysis

This notebook analyzes the CSV files produced by `experiment.py pcg-screen`.

It is designed to be placed in the repository, for example as:

```text
results_analysis/pcg_analysis.ipynb
```

The notebook expects experiment CSVs in:

```text
results/pcg_screening.csv
results/medium_enemy_tuning.csv
```

The main questions are:

1. Does PCG create a useful difficulty curve across map sizes?
2. Which PCG variables have the largest effects on agent performance?
3. Which settings create useful skill separation between the heuristic agent and MCTS?
4. Are there surprising interactions between enemy count, item count, wall density, and map type?
5. Which PCG defaults should be validated next?

In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_DIR = Path("results")
OUT_DIR = Path("results_analysis/pcg_analysis")
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

FULL_SCREEN_CSV = RESULTS_DIR / "pcg_screening.csv"
MEDIUM_TUNING_CSV = RESULTS_DIR / "medium_enemy_tuning.csv"

AGENT_ORDER = ["random", "heuristic", "mcts_small", "mcts_medium"]
SIZE_ORDER = ["small", "medium", "large"]
MAP_ORDER = ["baseline", "random_walk", "arena"]
ENEMY_ORDER = ["light", "normal", "medium_plus", "heavy"]
ITEM_ORDER = ["few", "normal", "many"]
WALL_ORDER = ["open", "normal"]

In [ ]:
def load_results(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Run the experiment first or update the path."
        )

    df = pd.read_csv(path)
    df = df.copy()

    df["won"] = df["won"].astype(int)

    numeric_cols = [
        "turns",
        "damage_taken",
        "final_hp",
        "remaining_enemies",
        "chokepoint_count",
        "reachable_floor_fraction",
        "generated_enemy_count",
        "generated_obstacle_density",
        "interior_wall_density",
        "dead_end_count",
        "avg_enemy_attack_distance",
        "max_enemy_attack_distance",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


full = load_results(FULL_SCREEN_CSV)
medium = load_results(MEDIUM_TUNING_CSV)

print("Full PCG screen:")
print(f"  rows: {len(full):,}")
print(f"  cells: {full['cell_id'].nunique() if 'cell_id' in full.columns else 'n/a'}")
print(f"  episodes per cell: {full.groupby('cell_id').size().iloc[0] if 'cell_id' in full.columns else 'n/a'}")

print("\nMedium enemy tuning:")
print(f"  rows: {len(medium):,}")
print(f"  cells: {medium['cell_id'].nunique() if 'cell_id' in medium.columns else 'n/a'}")
print(f"  episodes per cell: {medium.groupby('cell_id').size().iloc[0] if 'cell_id' in medium.columns else 'n/a'}")

## 1. Overall difficulty curve

The first check is whether generated maps become harder as the map size increases.
A good progression should look like:

```text
small  -> easy/tutorial
medium -> balanced/main setting
large  -> hard but not impossible
```

In [ ]:
overall = (
    full.groupby(["size", "agent"])["won"]
    .mean()
    .unstack()
    .reindex(index=SIZE_ORDER, columns=AGENT_ORDER)
)

display(overall.style.format("{:.1%}"))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(overall.index))
width = 0.8 / len(AGENT_ORDER)

for i, agent in enumerate(AGENT_ORDER):
    ax.bar(x + (i - (len(AGENT_ORDER) - 1) / 2) * width, overall[agent], width, label=agent)

ax.set_xticks(x)
ax.set_xticklabels(overall.index)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Win rate")
ax.set_title("Win rate by map size and agent")
ax.legend()

for i, size in enumerate(overall.index):
    for j, agent in enumerate(AGENT_ORDER):
        value = overall.loc[size, agent]
        if pd.notna(value):
            ax.text(
                i + (j - (len(AGENT_ORDER) - 1) / 2) * width,
                value + 0.02,
                f"{value:.0%}",
                ha="center",
                fontsize=8,
            )

fig.tight_layout()
fig.savefig(FIG_DIR / "01_winrate_by_size_and_agent.png", dpi=180)
plt.show()

## 2. Which variables have the biggest effects?

This section computes a simple main-effect estimate: for each variable, how large is the win-rate swing between its easiest and hardest levels?

This is not a full statistical model, but it is useful for quickly identifying the strongest balancing levers.

In [ ]:
variables = ["size", "enemy_profile", "item_profile", "wall_profile", "map_type"]

effect_rows = []
for agent in ["heuristic", "mcts_medium"]:
    df_agent = full[full["agent"] == agent]

    for variable in variables:
        means = df_agent.groupby(variable)["won"].mean()
        effect_rows.append(
            {
                "agent": agent,
                "variable": variable,
                "min_win_rate": means.min(),
                "max_win_rate": means.max(),
                "effect_range": means.max() - means.min(),
            }
        )

effects = pd.DataFrame(effect_rows)
display(effects.sort_values(["agent", "effect_range"], ascending=[True, False]).style.format({
    "min_win_rate": "{:.1%}",
    "max_win_rate": "{:.1%}",
    "effect_range": "{:.1%}",
}))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(variables))
width = 0.35

for i, agent in enumerate(["heuristic", "mcts_medium"]):
    values = effects[effects["agent"] == agent].set_index("variable").loc[variables, "effect_range"]
    ax.bar(x + (i - 0.5) * width, values, width, label=agent)

ax.set_xticks(x)
ax.set_xticklabels(variables, rotation=25, ha="right")
ax.set_ylabel("Win-rate swing")
ax.set_title("Main-effect range by PCG variable")
ax.legend()

for i, variable in enumerate(variables):
    for j, agent in enumerate(["heuristic", "mcts_medium"]):
        value = effects[(effects["agent"] == agent) & (effects["variable"] == variable)]["effect_range"].iloc[0]
        ax.text(i + (j - 0.5) * width, value + 0.015, f"{value:.0%}", ha="center", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "02_main_effect_ranges.png", dpi=180)
plt.show()

## 3. Medium-map enemy tuning

The full screen showed that medium maps needed a more precise enemy-count setting. The targeted experiment compares:

- `normal`
- `medium_plus`
- `heavy`

on medium maps only.

In [ ]:
medium_enemy = (
    medium.groupby(["enemy_profile", "agent"])["won"]
    .mean()
    .unstack()
    .reindex(index=["normal", "medium_plus", "heavy"], columns=AGENT_ORDER)
)

display(medium_enemy.style.format("{:.1%}"))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(medium_enemy.index))
width = 0.8 / len(AGENT_ORDER)

for i, agent in enumerate(AGENT_ORDER):
    ax.bar(x + (i - (len(AGENT_ORDER) - 1) / 2) * width, medium_enemy[agent], width, label=agent)

ax.set_xticks(x)
ax.set_xticklabels(medium_enemy.index)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Win rate")
ax.set_title("Medium-map tuning: enemy profile")
ax.legend()

for i, enemy in enumerate(medium_enemy.index):
    for j, agent in enumerate(AGENT_ORDER):
        value = medium_enemy.loc[enemy, agent]
        if pd.notna(value):
            ax.text(
                i + (j - (len(AGENT_ORDER) - 1) / 2) * width,
                value + 0.02,
                f"{value:.0%}",
                ha="center",
                fontsize=8,
            )

fig.tight_layout()
fig.savefig(FIG_DIR / "03_medium_enemy_tuning.png", dpi=180)
plt.show()

## 4. Skill separation

A useful balancing signal is the difference between MCTS-medium and the heuristic agent:

```text
skill gap = win_rate(mcts_medium) - win_rate(heuristic)
```

A positive gap means search/planning gives an advantage. Very small gaps usually mean the setting is either too easy or does not reward planning much.

In [ ]:
med_summary = (
    medium.groupby(["map_type", "enemy_profile", "item_profile", "wall_profile", "agent"])["won"]
    .mean()
    .reset_index()
)

pivot = med_summary.pivot_table(
    index=["map_type", "enemy_profile", "item_profile", "wall_profile"],
    columns="agent",
    values="won",
    aggfunc="first",
).reset_index()

pivot["skill_gap"] = pivot["mcts_medium"] - pivot["heuristic"]

# Heatmaps per wall profile
for wall_profile in WALL_ORDER:
    subset = pivot[pivot["wall_profile"] == wall_profile]

    rows = []
    for map_type in MAP_ORDER:
        for enemy in ["normal", "medium_plus", "heavy"]:
            rows.append((map_type, enemy))

    matrix = []
    row_labels = []
    for map_type, enemy in rows:
        row_labels.append(f"{map_type}\n{enemy}")
        row_values = []
        for item in ["normal", "many"]:
            match = subset[
                (subset["map_type"] == map_type)
                & (subset["enemy_profile"] == enemy)
                & (subset["item_profile"] == item)
            ]
            row_values.append(match["skill_gap"].iloc[0] if len(match) else np.nan)
        matrix.append(row_values)

    matrix = np.array(matrix, dtype=float)

    fig, ax = plt.subplots(figsize=(7, 8))
    im = ax.imshow(matrix, vmin=-0.2, vmax=0.8)
    ax.set_xticks(np.arange(2))
    ax.set_xticklabels(["normal items", "many items"])
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.set_title(f"MCTS-medium advantage over heuristic\nmedium maps, walls={wall_profile}")

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if not np.isnan(matrix[i, j]):
                ax.text(j, i, f"{matrix[i, j]:+.0%}", ha="center", va="center")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"04_skill_gap_medium_{wall_profile}.png", dpi=180)
    plt.show()

## 5. Interaction: wall profile and map type

Wall profile has a small average effect, but it can matter in specific harder configurations.
This plot shows the effect of changing from `open` to `normal` walls on MCTS-medium win rate.

In [ ]:
mcts_med = medium[medium["agent"] == "mcts_medium"]

wall_effect = (
    mcts_med.groupby(["map_type", "enemy_profile", "wall_profile"])["won"]
    .mean()
    .unstack()
    .reset_index()
)

wall_effect = wall_effect.dropna(subset=["open", "normal"])
wall_effect["normal_minus_open"] = wall_effect["normal"] - wall_effect["open"]
wall_effect["label"] = wall_effect["map_type"] + "\n" + wall_effect["enemy_profile"]

display(wall_effect[["map_type", "enemy_profile", "open", "normal", "normal_minus_open"]].style.format({
    "open": "{:.1%}",
    "normal": "{:.1%}",
    "normal_minus_open": "{:+.1%}",
}))

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(wall_effect))
ax.bar(x, wall_effect["normal_minus_open"])
ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(wall_effect["label"], rotation=35, ha="right")
ax.set_ylabel("Win-rate change")
ax.set_title("MCTS-medium: normal walls minus open walls")

for i, value in enumerate(wall_effect["normal_minus_open"]):
    offset = 0.015 if value >= 0 else -0.05
    ax.text(i, value + offset, f"{value:+.0%}", ha="center", fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / "05_wall_profile_interaction.png", dpi=180)
plt.show()

## 6. Interaction: chokepoints and pacing

Random-walk maps are not necessarily impossible, but they often have many chokepoints and longer games.
This is a pacing issue rather than only a win-rate issue.

In [ ]:
summary = (
    full[full["agent"].isin(["heuristic", "mcts_medium"])]
    .groupby(["size", "map_type", "enemy_profile", "item_profile", "wall_profile", "agent"])
    .agg(
        win_rate=("won", "mean"),
        avg_turns=("turns", "mean"),
        avg_chokepoints=("chokepoint_count", "mean"),
        avg_damage=("damage_taken", "mean"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 6))

for map_type in MAP_ORDER:
    subset = summary[(summary["agent"] == "mcts_medium") & (summary["map_type"] == map_type)]
    ax.scatter(subset["avg_chokepoints"], subset["avg_turns"], label=map_type, alpha=0.75)

ax.set_xlabel("Average chokepoints")
ax.set_ylabel("Average turns")
ax.set_title("Chokepoints vs. MCTS-medium pacing")
ax.legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "06_chokepoints_vs_turns.png", dpi=180)
plt.show()

## 7. Rank candidate configurations

The balance score below is only a ranking aid. It favors:

- high but not trivial MCTS-medium win rate,
- non-perfect heuristic win rate,
- positive MCTS-vs-heuristic skill gap,
- low random-agent success.

Use this to identify promising settings, then interpret them manually.

In [ ]:
CONFIG_COLS = ["size", "map_type", "enemy_profile", "item_profile", "wall_profile"]

def summarize_configs(df: pd.DataFrame) -> pd.DataFrame:
    by_agent = (
        df.groupby(CONFIG_COLS + ["agent"])
        .agg(
            win_rate=("won", "mean"),
            avg_turns=("turns", "mean"),
            avg_damage=("damage_taken", "mean"),
            avg_chokepoints=("chokepoint_count", "mean"),
        )
        .reset_index()
    )

    pivot = by_agent.pivot_table(
        index=CONFIG_COLS,
        columns="agent",
        values=["win_rate", "avg_turns", "avg_damage", "avg_chokepoints"],
        aggfunc="first",
    )

    pivot.columns = [f"{metric}_{agent}" for metric, agent in pivot.columns]
    pivot = pivot.reset_index()

    for agent in AGENT_ORDER:
        col = f"win_rate_{agent}"
        if col not in pivot.columns:
            pivot[col] = np.nan

    pivot["skill_gap_mcts_medium_vs_heuristic"] = (
        pivot["win_rate_mcts_medium"] - pivot["win_rate_heuristic"]
    )
    pivot["search_gain_mcts_medium_vs_small"] = (
        pivot["win_rate_mcts_medium"] - pivot["win_rate_mcts_small"]
    )

    m = pivot["win_rate_mcts_medium"]
    h = pivot["win_rate_heuristic"]
    r = pivot["win_rate_random"]

    pivot["balance_score"] = (
        1.0
        - (m - 0.80).abs()
        - 0.7 * (h - 0.50).abs()
        + 0.6 * (m - h)
        - 0.5 * r.fillna(0)
    )

    return pivot.sort_values("balance_score", ascending=False)


full_config_summary = summarize_configs(full)
medium_config_summary = summarize_configs(medium)

full_config_summary.to_csv(TABLE_DIR / "full_config_summary.csv", index=False)
medium_config_summary.to_csv(TABLE_DIR / "medium_config_summary.csv", index=False)

display(
    medium_config_summary[
        [
            "size",
            "map_type",
            "enemy_profile",
            "item_profile",
            "wall_profile",
            "win_rate_random",
            "win_rate_heuristic",
            "win_rate_mcts_small",
            "win_rate_mcts_medium",
            "skill_gap_mcts_medium_vs_heuristic",
            "balance_score",
        ]
    ]
    .head(20)
    .style.format({
        "win_rate_random": "{:.1%}",
        "win_rate_heuristic": "{:.1%}",
        "win_rate_mcts_small": "{:.1%}",
        "win_rate_mcts_medium": "{:.1%}",
        "skill_gap_mcts_medium_vs_heuristic": "{:+.1%}",
        "balance_score": "{:.2f}",
    })
)

top = medium_config_summary.head(15).copy()
top["label"] = (
    top["map_type"]
    + " | "
    + top["enemy_profile"]
    + " | "
    + top["item_profile"]
    + " | "
    + top["wall_profile"]
)

fig, ax = plt.subplots(figsize=(11, 6))
y = np.arange(len(top))
ax.barh(y, top["balance_score"])
ax.set_yticks(y)
ax.set_yticklabels(top["label"])
ax.invert_yaxis()
ax.set_xlabel("Balance score")
ax.set_title("Top medium-map candidate configurations")

fig.tight_layout()
fig.savefig(FIG_DIR / "07_top_medium_candidates.png", dpi=180)
plt.show()

## 8. Recommended PCG defaults from these experiments

Based on the current results:

| Size | Map type | Enemy profile | Item profile | Wall profile |
|---|---|---|---|---|
| small | all | normal | normal | normal |
| medium | baseline | medium_plus | many | open |
| medium | random_walk | medium_plus | many | open |
| medium | arena | medium_plus | many | open |
| large | baseline | normal | many | normal |
| large | random_walk | normal | many | open |
| large | arena | normal | many | open |

Rationale:

- `small` is already easy and can act as a tutorial setting.
- `medium_plus` creates useful skill separation on medium maps.
- `many` items keep harder maps fair and reward planning.
- `open` walls are preferred for medium maps and random-walk pacing.
- Large maps should generally avoid `heavy` enemies because they become too punishing.

The next experiment should be a final validation run with only these selected defaults.